# Rung 2 — excess chemical potential against Carnahan–Starling

**What this validates:** the weighted densities $\xi_0..\xi_3$, the White-Bear/BMCSL
free energy, the exact integer table, **and the self-exclusion term**.

**Analytic answer:** at equilibrium the total chemical potential is uniform in space,

$$\underbrace{\ln \rho(x)}_{\mu_{\rm ideal}} + \mu_{\rm ex}(\rho(x)) + \phi(x) = \text{const},$$

so the excess chemical potential can be read straight off a measured density profile:

$$\mu_{\rm ex}(\rho(x)) = \text{const} - \ln \rho(x) - \phi(x).$$

For a single species BMCSL reduces to Carnahan–Starling,

$$\mu_{\rm ex}(\eta) = \frac{8\eta - 9\eta^2 + 3\eta^3}{(1-\eta)^3},$$

which is the curve we compare against. Only one free parameter — the additive constant —
is fitted; the *shape* is parameter-free.

**Why this tests self-exclusion.** A hopping particle must not feel its own volume at
the voxel it leaves. Omitting that term changes the shape of the recovered
$\mu_{\rm ex}(\eta)$, not merely its offset, so it cannot hide in the fitted constant.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from vex_rddme import Simulation, Species, mu_ex_carnahan_starling
from vex_rddme.guards import suggest_tau
from vex_rddme import viz
from vex_rddme.observe import (
    Series, project, mu_ex_from_profile, align_additive_constant,
    relative_discrepancy, report_comparison, QuotientAccumulator,
)

## Setup

The timestep is **not** set by the bare CFL bound here. With exclusion on, the
constraint is the downhill Bernoulli factor: $2\,d\,q\,\mu_{\rm ex}(\eta_{\max}) \le 1$.
At $\eta = 0.5$, $\mu_{\rm ex} \approx 17\,k_BT$, an order of magnitude tighter than
CFL. `suggest_tau` computes it from the packing fraction you expect to reach.

In [ ]:
SHAPE     = (32, 24)
VOXEL_NM  = 20.0
SIGMA_NM  = 8.0
CAP       = 20
GAMMA     = 3.0
N_PER_VOXEL = 3          # seeded exactly, not randomly: no high-occupancy outliers
N_STEPS   = 60_000
BURN_IN   = N_STEPS // 3
SAMPLE_EVERY = 25

dxi3 = (np.pi / 6) * SIGMA_NM ** 3 / VOXEL_NM ** 3
eta_peak_guess = 3.0 * N_PER_VOXEL * dxi3        # allow the field to triple the peak
TAU_S = suggest_tau(D_um2_s=1.0, voxel_nm=VOXEL_NM, dim=2, eta_max=eta_peak_guess)

print(f"one particle contributes dxi3 = {dxi3:.5f}")
print(f"mean packing fraction        = {N_PER_VOXEL * dxi3:.4f}")
print(f"assumed peak                 = {eta_peak_guess:.4f}")
print(f"suggested tau                = {TAU_S:.3e} s   (q = {1e6 * TAU_S / VOXEL_NM**2:.4f})")

ramp = np.arange(SHAPE[-1], dtype=float) / SHAPE[-1]
psi  = np.broadcast_to(ramp, SHAPE).copy()[None, ...]

sim = Simulation(
    shape=SHAPE, voxel_nm=VOXEL_NM,
    species=[Species("A", sigma_nm=SIGMA_NM, gamma=np.array([GAMMA]))],
    occupancy_cap=CAP, psi=psi, D_um2_s=1.0, tau_s=TAU_S, seed=0,
)
sim.set_counts("A", np.full(SHAPE, N_PER_VOXEL, dtype=np.int64))
sim.record_initial()
sim

The construction guards have already reported above: the σ/h consistency check,
the maximum attainable packing fraction, the table that was built, whether exclusion or
the occupancy cap is the binding constraint, and the CFL margin.

In [ ]:
rho = Series("density")
n_rows = SHAPE[0]

for i in range(N_STEPS):
    sim.step()
    if i >= BURN_IN and (i - BURN_IN) % SAMPLE_EVERY == 0:
        rho.add(project(sim.state.lattice_view("A"), sim.lattice) / n_rows)

sim.state.check_mass()
print(f"{rho.n} samples;  density {rho.mean.min():.2f} - {rho.mean.max():.2f} per voxel")

## Extract μ_ex and compare

In [ ]:
density  = rho.mean
eta      = density * dxi3
phi      = GAMMA * ramp

analytic = mu_ex_carnahan_starling(eta)
measured = align_additive_constant(mu_ex_from_profile(density, phi), analytic)

# error bars propagate from the density: d(mu_ex)/d(rho) = -1/rho
sem = rho.sem / density

print(f"packing fraction spans {eta.min():.3f} - {eta.max():.3f}")
print(f"mu_ex spans            {analytic.min():.3f} - {analytic.max():.3f} kT")
print()
print(report_comparison("mu_ex(eta) vs Carnahan-Starling", measured, analytic, sem=sem))

VERDICTS = {"mu_ex vs Carnahan-Starling": relative_discrepancy(measured, analytic)["max"]}

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
viz.plot_mu_ex(eta, measured, analytic, sem=sem, ax=ax[0])
ax[0].set_title("Rung 2: excess chemical potential", fontsize=10)
viz.plot_profile(density, sem=rho.sem, ax=ax[1],
                 xlabel="voxel along the field axis",
                 ylabel="mean occupancy per voxel",
                 label_measured="measured density",
                 title="the profile it was extracted from")
plt.tight_layout(); plt.show()

## What to take from this

The excess chemical potential recovered from a simulated density profile matches
Carnahan–Starling across the sampled packing-fraction range, with one fitted additive
constant and no other free parameters. That exercises the weighted densities, the BMCSL
functional, the integer table, and the self-exclusion term together.

**Try changing:**

- `GAMMA = 0` — the profile goes flat, the packing-fraction range collapses, and there
  is nothing left to compare. `relative_discrepancy` will refuse a flat prediction.
- `N_PER_VOXEL = 8` — denser, so $\mu_{\rm ex}$ is larger and more sharply curved.
  `suggest_tau` will return a smaller timestep, and the run will take proportionally
  longer.
- `SIGMA_NM = 20` with `VOXEL_NM = 20` — rejected at construction. A sphere as wide as
  the voxel admits at most one particle per voxel, so no crowding study is possible;
  the guard says so and names the fix.